# Analítica de Plataforma IoT (Synapse Serverless)

Este notebook consulta la capa `Gold` y `Silver` expuestas a través de Azure Synapse Serverless SQL Pool. Ejecutaremos las 3 consultas analíticas requeridas para la validación del proyecto.

## 1. Tendencia Temporal (Promedio histórico por dispositivo por hora)

In [ ]:
%%sql
SELECT 
    device_id,
    CAST(window_end AS DATE) as date_value,
    DATEPART(HOUR, window_end) as hour_value,
    AVG(avg_temperature) as mean_hourly_temp,
    MAX(max_temperature) as max_hourly_temp
FROM 
    OPENROWSET(
        BULK 'https://stdataplatformdev01.dfs.core.windows.net/gold/telemetry/**/*.parquet',
        FORMAT = 'PARQUET'
    ) AS [gold_telemetry]
GROUP BY 
    device_id,
    CAST(window_end AS DATE),
    DATEPART(HOUR, window_end)
ORDER BY 
    date_value DESC, 
    hour_value DESC, 
    device_id;

## 2. Top-N Dispositivos (Top 5 con más alertas enviadas)

*Utilizamos los datos que salieron nativos desde la rama de Alertas de Stream Analytics hacia la capa Silver.*

In [ ]:
%%sql
WITH SilverAlerts AS (
    SELECT
        JSON_VALUE(jsonContent, '$.device_id') as device_id,
        CAST(JSON_VALUE(jsonContent, '$.temperature') AS FLOAT) as temperature,
        JSON_VALUE(jsonContent, '$.status') as status,
        CAST(JSON_VALUE(jsonContent, '$.event_time') AS DATETIME2) as event_time
    FROM
        OPENROWSET(
            BULK 'https://stdataplatformdev01.dfs.core.windows.net/silver/alerts/**/*.json',
            FORMAT = 'CSV',
            FIELDQUOTE = '0x0b',
            FIELDTERMINATOR ='0x0b',
            ROWTERMINATOR = '0x0b'
        )
        WITH ( jsonContent varchar(MAX) ) AS [result]
)
SELECT TOP 5
    device_id,
    COUNT(*) as total_alerts,
    MAX(temperature) as peak_temperature_registered
FROM 
    SilverAlerts
WHERE 
    status = 'WARNING' 
    AND event_time >= DATEADD(day, -7, GETDATE())
GROUP BY 
    device_id
ORDER BY 
    total_alerts DESC;

## 3. Agregaciones Generales (Métricas base del parque IoT)

In [ ]:
%%sql
SELECT
    device_id,
    MIN(avg_temperature) as historical_min_temp,
    MAX(max_temperature) as historical_max_temp,
    AVG(avg_humidity) as avg_overall_humidity,
    SUM(total_events_in_minute) as total_readings_captured
FROM
    OPENROWSET(
        BULK 'https://stdataplatformdev01.dfs.core.windows.net/gold/telemetry/**/*.parquet',
        FORMAT = 'PARQUET'
    ) AS [gold_telemetry]
GROUP BY
    device_id;